<h1 style="font-family: 'poppins'; font-weight: bold; color: Blue;">👨‍💻Author: Muhammad Faheem Iqbal</h1>

[![GitHub](https://img.shields.io/badge/GitHub-Profile-blue?style=for-the-badge&logo=github)](https://github.com/FaheemAI1024)
[![Kaggle](https://img.shields.io/badge/Kaggle-Profile-blue?style=for-the-badge&logo=kaggle)](https://www.kaggle.com/faheem113141) 
[![LinkedIn](https://img.shields.io/badge/LinkedIn-Profile-blue?style=for-the-badge&logo=linkedin)](https://www.linkedin.com/in/muhammad-faheem-iqbal-ai-solutions/)  

[![Facebook](https://img.shields.io/badge/Facebook-Profile-blue?style=for-the-badge&logo=facebook)](https://www.facebook.com/share/1D6B9ZA4XY/) 
[![TikTok](https://img.shields.io/badge/TikTok-Profile-blue?style=for-the-badge&logo=tiktok)](https://www.tiktok.com/@data_scientist04?_t=8kW2bLg8CFl&_r=1)
[![HuggingFace](https://img.shields.io/badge/huggingface-Profile-yellow?style=for-the-badge&logo=huggingface)](https://huggingface.co/FaheemAi1024)

[![Twitter/X](https://img.shields.io/badge/Twitter-Profile-blue?style=for-the-badge&logo=twitter)](https://x.com/MFaheem113141?t=__88BWMyKGZcC08sw3SJtA&s=09) 
[![Instagram](https://img.shields.io/badge/Instagram-Profile-blue?style=for-the-badge&logo=instagram)](https://www.instagram.com/i_am_faheeeem?igsh=MXhlcG0zdTZ6Mnl5Yw==) 
[![Email](https://img.shields.io/badge/Email-Contact%20Me-red?style=for-the-badge&logo=email)](mailto:faheemiqbalbwn2002@gmail.com)


In [1]:
print("helo")

helo


In [2]:
import requests
import pandas as pd
import time  # for small delay to be polite to the API

# Function to fetch time-series data for one indicator
def fetch_indicator_data(indicator_code, countries='all', years='2000:2024'):
    url = f"https://api.worldbank.org/v2/country/{countries}/indicator/{indicator_code}?date={years}&format=json&per_page=20000"
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()  # Raise error for bad status codes
        data = response.json()
        
        if len(data) > 1 and isinstance(data[1], list) and data[1]:
            records = []
            for item in data[1]:
                records.append({
                    'country': item['country']['value'],
                    'countryiso3code': item.get('countryiso3code', ''),
                    'date': item['date'],
                    'value': item['value'],
                    'indicator_code': item['indicator']['id'],
                    'indicator': item['indicator']['value'],  # official name
                    'unit': item.get('unit', ''),
                    'decimal': item.get('decimal', None)
                })
            df = pd.DataFrame(records)
            return df
        else:
            print(f"No data found for indicator {indicator_code}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {indicator_code}: {e}")
        return None

# Key indicators for SAP Analytics Planning (economic forecasting, scenarios, storytelling)
key_indicators = {
    'NY.GDP.MKTP.CD': 'GDP (current US$)',
    'NY.GDP.MKTP.KD.ZG': 'GDP growth (annual %)',
    'NY.GDP.PCAP.CD': 'GDP per capita (current US$)',               # Added - very useful for planning
    'FP.CPI.TOTL.ZG': 'Inflation, consumer prices (annual %)',
    'SL.UEM.TOTL.ZS': 'Unemployment, total (% of total labor force)',
    'SP.POP.TOTL': 'Population, total',
    'SP.POP.GROW': 'Population growth (annual %)',                  # Added - good for demographic trends
    'NE.EXP.GNFS.CD': 'Exports of goods and services (current US$)',
    'NE.IMP.GNFS.CD': 'Imports of goods and services (current US$)',
    'NE.TRD.GNFS.ZS': 'Trade (% of GDP)',                           # Added - trade openness
    'EG.USE.PCAP.KG.OE': 'Energy use (kg of oil equivalent per capita)',
    'EN.ATM.CO2E.KT': 'CO2 emissions (kt)',
    'EN.ATM.CO2E.PC': 'CO2 emissions (metric tons per capita)',     # Added - per capita view
    'SL.GDP.PCAP.EM.KD': 'GDP per person employed (constant 2017 PPP $)'
}

# Main execution
if __name__ == "__main__":
    print("Starting to fetch World Bank data for SAP Analytics planning model...\n")
    
    all_dataframes = []
    
    for code, custom_name in key_indicators.items():
        print(f"Fetching: {custom_name} ({code}) ...")
        df = fetch_indicator_data(code, countries='all', years='2000:2024')
        if df is not None:
            df['indicator_name'] = custom_name  # your friendly name for SAC
            all_dataframes.append(df)
            print(f"   → Got {len(df)} rows")
        time.sleep(1.2)  # polite delay to avoid rate limiting
    
    if all_dataframes:
        # Combine all into one big table
        combined_df = pd.concat(all_dataframes, ignore_index=True)
        
        # Optional: sort for better readability (by country, indicator, year descending)
        combined_df['date'] = pd.to_numeric(combined_df['date'], errors='coerce')
        combined_df = combined_df.sort_values(['country', 'indicator_code', 'date'], ascending=[True, True, False])
        
        # Save to single CSV
        output_file = 'worldbank_economic_planning_data.csv'
        combined_df.to_csv(output_file, index=False)
        
        print(f"\nSuccess! Combined dataset saved to '{output_file}'")
        print(f"Total rows: {len(combined_df):,}")
        print(f"Unique countries: {combined_df['country'].nunique()}")
        print(f"Unique indicators: {combined_df['indicator_code'].nunique()}")
        print(f"Time range: {combined_df['date'].min()} – {combined_df['date'].max()}")
        
        print("\nColumns in the CSV:")
        print(", ".join(combined_df.columns))
        
        print("\nNext steps in SAP Analytics Cloud:")
        print("1. Import this CSV → New Model (or upload to existing dataset)")
        print("2. Map dimensions: Country, date (as Time dimension), indicator_name / indicator_code")
        print("3. Measures: value (numeric)")
        print("4. Build planning model → versions (Actual / Budget / Forecast)")
        print("5. Use Predictive Forecast or Smart Predict on time-series (e.g. GDP growth, inflation)")
        print("6. Create Stories → visualizations, what-if scenarios, value driver trees")
    else:
        print("\nNo data was fetched. Check internet connection or API status.")

Starting to fetch World Bank data for SAP Analytics planning model...

Fetching: GDP (current US$) (NY.GDP.MKTP.CD) ...
   → Got 6650 rows
Fetching: GDP growth (annual %) (NY.GDP.MKTP.KD.ZG) ...
   → Got 6650 rows
Fetching: GDP per capita (current US$) (NY.GDP.PCAP.CD) ...
   → Got 6650 rows
Fetching: Inflation, consumer prices (annual %) (FP.CPI.TOTL.ZG) ...
   → Got 6650 rows
Fetching: Unemployment, total (% of total labor force) (SL.UEM.TOTL.ZS) ...
   → Got 6650 rows
Fetching: Population, total (SP.POP.TOTL) ...
   → Got 6650 rows
Fetching: Population growth (annual %) (SP.POP.GROW) ...
   → Got 6650 rows
Fetching: Exports of goods and services (current US$) (NE.EXP.GNFS.CD) ...
   → Got 6650 rows
Fetching: Imports of goods and services (current US$) (NE.IMP.GNFS.CD) ...
   → Got 6650 rows
Fetching: Trade (% of GDP) (NE.TRD.GNFS.ZS) ...
   → Got 6650 rows
Fetching: Energy use (kg of oil equivalent per capita) (EG.USE.PCAP.KG.OE) ...
   → Got 6650 rows
Fetching: CO2 emissions (kt) (

In [3]:
df = pd.read_csv("worldbank_economic_planning_data.csv")

df.head(10)

,country,countryiso3code,date,value,indicator_code,indicator,unit,decimal,indicator_name
0,Afghanistan,AFG,2024,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)
1,Afghanistan,AFG,2023,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)
2,Afghanistan,AFG,2022,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)
3,Afghanistan,AFG,2021,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)
4,Afghanistan,AFG,2020,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)
5,Afghanistan,AFG,2019,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)
6,Afghanistan,AFG,2018,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)
7,Afghanistan,AFG,2017,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)
8,Afghanistan,AFG,2016,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)
9,Afghanistan,AFG,2015,NaN,EG.USE.PCAP.KG.OE,Energy use (kg of oil equivalent per capita),NaN,0,Energy use (kg of oil equivalent per capita)


In [4]:
df.tail()

,country,countryiso3code,date,value,indicator_code,indicator,unit,decimal,indicator_name
79795,Zimbabwe,ZWE,2004,12365896.0,SP.POP.TOTL,"Population, total",NaN,0,"Population, total"
79796,Zimbabwe,ZWE,2003,12232323.0,SP.POP.TOTL,"Population, total",NaN,0,"Population, total"
79797,Zimbabwe,ZWE,2002,12087653.0,SP.POP.TOTL,"Population, total",NaN,0,"Population, total"
79798,Zimbabwe,ZWE,2001,11971901.0,SP.POP.TOTL,"Population, total",NaN,0,"Population, total"
79799,Zimbabwe,ZWE,2000,11892055.0,SP.POP.TOTL,"Population, total",NaN,0,"Population, total"


In [5]:
df.size

718200

In [6]:
df.shape

(79800, 9)

In [7]:
df.isnull()

,country,countryiso3code,date,value,indicator_code,indicator,unit,decimal,indicator_name
0,False,False,False,True,False,False,True,False,False
1,False,False,False,True,False,False,True,False,False
2,False,False,False,True,False,False,True,False,False
3,False,False,False,True,False,False,True,False,False
4,False,False,False,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...
79795,False,False,False,False,False,False,True,False,False
79796,False,False,False,False,False,False,True,False,False
79797,False,False,False,False,False,False,True,False,False
79798,False,False,False,False,False,False,True,False,False


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79800 entries, 0 to 79799
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   country          79800 non-null  object 
 1   countryiso3code  78300 non-null  object 
 2   date             79800 non-null  int64  
 3   value            70765 non-null  float64
 4   indicator_code   79800 non-null  object 
 5   indicator        79800 non-null  object 
 6   unit             0 non-null      float64
 7   decimal          79800 non-null  int64  
 8   indicator_name   79800 non-null  object 
dtypes: float64(2), int64(2), object(5)
memory usage: 5.5+ MB


In [10]:
df.describe()

,date,value,unit,decimal
count,79800.000000,7.076500e+04,0.0,79800.000000
mean,2012.000000,3.113928e+11,NaN,0.416667
std,7.211148,2.708737e+12,NaN,0.493010
min,2000.000000,-5.440209e+01,NaN,0.000000
25%,2006.000000,5.589000e+00,NaN,0.000000
50%,2012.000000,2.378083e+03,NaN,0.000000
75%,2018.000000,3.914172e+08,NaN,1.000000
max,2024.000000,1.109827e+14,NaN,1.000000
